# Thực hành: Explainable ML cho Dự đoán Customer Churn (Online Retail II)

Notebook này đi kèm file lý thuyết `nen-tang-kien-thuc-explainable-ml-churn.md` — mỗi phần code dưới đây tương ứng với một phần lý thuyết đã học. Khuyến nghị đọc phần lý thuyết tương ứng trước, sau đó chạy từng cell theo thứ tự.

**Trước khi chạy:** tải file dữ liệu `online_retail_II.csv` (hoặc `.xlsx`) từ Kaggle/UCI, đặt vào thư mục `data/` cùng cấp với notebook này. Xem file `huong-dan-cai-dat-cong-cu.md` nếu chưa cài môi trường.

## 0. Import thư viện

*(tương ứng Phần 12 — Python & Công cụ lập trình)*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')


## 1. Nạp dữ liệu

*(tương ứng Phần 12.2 — thao tác pandas)*

In [ ]:
# Nếu file là .csv:
df = pd.read_csv('data/online_retail_II.csv', encoding='ISO-8859-1')

# Nếu file gốc là .xlsx với 2 sheet (Year 2009-2010, Year 2010-2011), dùng thay:
# df1 = pd.read_excel('data/online_retail_II.xlsx', sheet_name='Year 2009-2010')
# df2 = pd.read_excel('data/online_retail_II.xlsx', sheet_name='Year 2010-2011')
# df = pd.concat([df1, df2], ignore_index=True)

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df.shape)
df.head()


## 2. Làm sạch dữ liệu

*(tương ứng Phần 7 — Data Preprocessing, và phần Đặc điểm nghiệp vụ đã bàn trước đó)*

In [ ]:
# 1. Loại hóa đơn huỷ (Invoice bắt đầu bằng 'C')
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# 2. Loại các mã không phải sản phẩm thật (phí ship, điều chỉnh kế toán...)
non_product_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'DOT', 'ADJUST', 'ADJUST2', 'CRUK']
df = df[~df['StockCode'].astype(str).isin(non_product_codes)]

# 3. Loại dòng thiếu Customer ID (không gán được về một khách hàng cụ thể)
df = df.dropna(subset=['Customer ID'])

# 4. Loại Quantity hoặc Price <= 0
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# 5. Tính doanh thu từng dòng
df['Revenue'] = df['Quantity'] * df['Price']

print(f"Số dòng sau khi làm sạch: {len(df):,}")
print(f"Số khách hàng duy nhất: {df['Customer ID'].nunique():,}")
print(f"Khoảng thời gian: {df['InvoiceDate'].min()} -> {df['InvoiceDate'].max()}")


## 3. EDA (Exploratory Data Analysis)

*(tương ứng Phần 6 — EDA chi tiết)*

In [ ]:
# Phân phối tổng chi tiêu theo khách hàng (univariate)
plt.figure(figsize=(10, 4))
customer_revenue = df.groupby('Customer ID')['Revenue'].sum()
sns.histplot(customer_revenue[customer_revenue < customer_revenue.quantile(0.95)], bins=50)
plt.title('Phân phối tổng chi tiêu theo khách hàng (đã cắt outlier 5% cao nhất)')
plt.xlabel('Tổng chi tiêu (Revenue)')
plt.show()

# Doanh thu theo tháng - kiểm tra tính thời vụ
monthly_revenue = df.set_index('InvoiceDate').resample('ME')['Revenue'].sum()
monthly_revenue.plot(figsize=(10, 4), title='Doanh thu theo tháng', marker='o')
plt.ylabel('Revenue')
plt.show()


## 4. Gán nhãn churn (cutoff-based labeling)

*(tương ứng Phần 3.3 — Classification-based approach, và phần đóng khung bài toán đã bàn trước đó)*

Ý tưởng: chọn một mốc **cutoff**, dùng dữ liệu **trước** cutoff để tính đặc trưng, và xem khách hàng có mua hàng trong **N ngày sau** cutoff hay không để gán nhãn.

In [ ]:
cutoff_date = pd.Timestamp('2011-06-01')
churn_window_days = 90  # đổi thành 180 để thử churn 6 tháng

# --- Kiểm tra right-censoring trước khi gán nhãn ---
max_date = df['InvoiceDate'].max()
required_end = cutoff_date + timedelta(days=churn_window_days)
assert required_end <= max_date, (
    f"Cutoff quá gần cuối dữ liệu! Cần dữ liệu đến {required_end}, "
    f"nhưng dữ liệu chỉ có đến {max_date}."
)

# Dữ liệu quan sát: trước cutoff (dùng để tính feature)
obs_df = df[df['InvoiceDate'] < cutoff_date]

# Dữ liệu tương lai: từ cutoff đến cutoff + churn_window (dùng để gán nhãn)
future_df = df[(df['InvoiceDate'] >= cutoff_date) &
               (df['InvoiceDate'] < required_end)]

customers_before = obs_df['Customer ID'].unique()
customers_active_future = set(future_df['Customer ID'].unique())

labels = pd.DataFrame({'Customer ID': customers_before})
labels['churn'] = labels['Customer ID'].apply(lambda x: 0 if x in customers_active_future else 1)

print(labels['churn'].value_counts(normalize=True).rename('tỷ lệ'))


## 5. Feature Engineering (RFM mở rộng)

*(tương ứng Phần 5 — Feature Engineering, và Phần 3.5 — RFM Framework)*

In [ ]:
rfm = obs_df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (cutoff_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Revenue', 'sum'),
    AvgOrderValue=('Revenue', 'mean'),
    NumProducts=('StockCode', 'nunique'),
).reset_index()

data = rfm.merge(labels, on='Customer ID')
print(data.shape)
data.head()


## 6. Chia tập train/test

*(tương ứng Phần 8.3 — Cross-validation; lưu ý: đây là ví dụ đơn giản với 1 mốc cutoff. Khi mở rộng lên nhiều mốc cutoff (rolling window), cần đảm bảo mốc dùng để train luôn xảy ra trước mốc dùng để test theo thời gian.)*

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'NumProducts']
X = data[feature_cols]
y = data['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


## 7. Mô hình baseline (Logistic Regression, Decision Tree)

*(tương ứng Phần 4.1 và 4.2)*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(class_weight='balanced', random_state=42)
log_reg.fit(X_train_scaled, y_train)

dtree = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dtree.fit(X_train, y_train)


## 8. Mô hình nâng cao (Random Forest, XGBoost)

*(tương ứng Phần 4.3 — Ensemble Methods)*

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42
)
xgb_model.fit(X_train, y_train)


## 9. Đánh giá mô hình

*(tương ứng Phần 8 — Model Evaluation)*

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

def evaluate(model, X_test_input, name):
    y_prob = model.predict_proba(X_test_input)[:, 1]
    y_pred = model.predict(X_test_input)
    print(f"--- {name} ---")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")
    print(f"PR-AUC : {average_precision_score(y_test, y_prob):.3f}")
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print()

evaluate(log_reg, X_test_scaled, "Logistic Regression")
evaluate(dtree, X_test, "Decision Tree")
evaluate(rf, X_test, "Random Forest")
evaluate(xgb_model, X_test, "XGBoost")


## 10. Explainability với SHAP

*(tương ứng Phần 2.3 — Các phương pháp XAI cụ thể)*

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Giải thích GLOBAL: đặc trưng nào quan trọng nhất trên toàn bộ tập test
shap.summary_plot(shap_values, X_test)


In [ ]:
# Giải thích LOCAL: vì sao MỘT khách hàng cụ thể (dòng đầu tiên) bị dự đoán như vậy
shap.force_plot(
    explainer.expected_value, shap_values[0], X_test.iloc[0], matplotlib=True
)


## 11. Kiểm định ý nghĩa thống kê giữa 2 mô hình

*(tương ứng Phần 8.4 — Statistical Significance Testing, và Phần 11.3 — p-value)*

In [ ]:
from sklearn.model_selection import StratifiedKFold
from scipy.stats import wilcoxon

def cross_val_auc(model_fn, X, y, n_splits=5, seeds=range(5)):
    scores = []
    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for train_idx, test_idx in skf.split(X, y):
            model = model_fn(seed)
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            prob = model.predict_proba(X.iloc[test_idx])[:, 1]
            scores.append(roc_auc_score(y.iloc[test_idx], prob))
    return np.array(scores)

rf_scores = cross_val_auc(
    lambda s: RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=s),
    X, y
)
xgb_scores = cross_val_auc(
    lambda s: xgb.XGBClassifier(n_estimators=200, max_depth=5, eval_metric='logloss', random_state=s),
    X, y
)

print(f"Random Forest AUC: {rf_scores.mean():.3f} +/- {rf_scores.std():.3f}")
print(f"XGBoost AUC      : {xgb_scores.mean():.3f} +/- {xgb_scores.std():.3f}")

stat, p_value = wilcoxon(rf_scores, xgb_scores)
print(f"Wilcoxon signed-rank test p-value: {p_value:.4f}")
print("=> Khác biệt có ý nghĩa thống kê (p<0.05)" if p_value < 0.05 else "=> Chưa đủ cơ sở kết luận khác biệt có ý nghĩa")


## Bước tiếp theo

- Thử lại toàn bộ pipeline với `churn_window_days = 180` (churn 6 tháng) để so sánh.
- Thử nhiều mốc `cutoff_date` khác nhau (rolling window) để kiểm tra độ nhạy của định nghĩa churn — xem Phần 3.3 và Phần 6 trong file lý thuyết.
- Bổ sung bộ dữ liệu thứ 2 nếu hướng tới công bố học thuật (xem Phần 9 trong file lý thuyết).